# Module 3.2: Asymmetric Cost Matrices in Plant Economics
### 🇵🇱 Asymetryczne Macierze Kosztów w Finansach Fabryki

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/03_02_cost_sensitive_learning.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Quantify real financial loss: why a missed defect costs 200x more than an operator false alarm.  
> 🇵🇱 **Streszczenie:** Policz realne straty finansowe: dlaczego przeoczenie wady kosztuje 200x więcej niż fałszywy alarm technika.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Asymmetric Business Penalties**: Cost(False Negative) = $10,000 (catastrophic engine breakdown), Cost(False Positive) = $50 (30-second manual check).
2. **Extreme Class Imbalance**: Flawless parts represent 99% of line output; high accuracy is trivial and misleading.
3. **Threshold Economics**: The optimal threshold minimizes expected monetary loss, not classification error count.


In [ ]:
# 1. Simulate Plant Defect Probabilities & Financial Matrix
np.random.seed(42)
n_parts = 2000
defect_ratio = 0.03  # 3% defective parts

y_true = (np.random.rand(n_parts) < defect_ratio).astype(int)

# Simulated model predicted probabilities
# Defective parts have higher scores on average, but with overlap
pred_probs = np.zeros(n_parts)
pred_probs[y_true == 0] = np.random.beta(a=1.5, b=8.0, size=(y_true == 0).sum())
pred_probs[y_true == 1] = np.random.beta(a=6.0, b=2.0, size=(y_true == 1).sum())

print(f"Generated {n_parts} parts: {(y_true == 1).sum()} defective, {(y_true == 0).sum()} healthy.")


In [ ]:
# 2. Sweep Thresholds and Compute Real Dollar Losses
thresholds = np.linspace(0.01, 0.95, 100)
cost_fn = 10000.0  # $10,000 per missed defect
cost_fp = 50.0     # $50 per false alarm

total_costs = []
recalls = []

for t in thresholds:
    y_pred = (pred_probs >= t).astype(int)
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    
    cost = fn * cost_fn + fp * cost_fp
    total_costs.append(cost)
    recalls.append(tp / ((y_true == 1).sum() or 1))

best_idx = np.argmin(total_costs)
optimal_t = thresholds[best_idx]
min_cost = total_costs[best_idx]
naive_cost = total_costs[np.argmin(np.abs(thresholds - 0.50))]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, total_costs, color='#ef4444', linewidth=2.5, label='Total Expected Financial Loss ($)')
plt.axvline(optimal_t, color='#10b981', linestyle='--', label=f'Cost-Optimal Threshold = {optimal_t:.2f} (Loss: ${min_cost:,.0f})')
plt.axvline(0.50, color='#64748b', linestyle=':', label=f'Academic 0.50 Default (Loss: ${naive_cost:,.0f})')
plt.title("Financial Loss vs Classification Decision Threshold", fontsize=13, fontweight='bold')
plt.xlabel("Decision Threshold (tau)")
plt.ylabel("Total Loss [USD]")
plt.legend()
plt.grid(True, alpha=0.4)
plt.show()
print(f"Optimal Threshold saves ${naive_cost - min_cost:,.0f} compared to the default 0.50 threshold!")


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Define cost weights: Cost(FN) = $10,000 vs Cost(FP) = $50 *(🇵🇱 Zdefiniuj wagi kosztów: Koszt(FN) = 40 000 PLN vs Koszt(FP) = 200 PLN)*
- [ ] **Task 2:** Calculate Expected Total Cost across all decision thresholds *(🇵🇱 Wylicz całkowity oczekiwany koszt dla każdego progu decyzyjnego)*
- [ ] **Task 3:** Pinpoint the cost-optimal operating threshold *(🇵🇱 Wyznacz optymalny ekonomicznie próg decyzyjny)*


In [ ]:
# Exercise Challenge:
# Calculate Recall and Precision at the optimal threshold
y_opt = (pred_probs >= optimal_t).astype(int)
tp_opt = ((y_true == 1) & (y_opt == 1)).sum()
fp_opt = ((y_true == 0) & (y_opt == 1)).sum()
fn_opt = ((y_true == 1) & (y_opt == 0)).sum()

recall_opt = tp_opt / (tp_opt + fn_opt)
precision_opt = tp_opt / (tp_opt + fp_opt)
print(f"Optimal Operating Point -> Recall: {recall_opt*100:.1f}%, Precision: {precision_opt*100:.1f}%")
assert recall_opt >= 0.90, "Recall must be at least 90% to prevent expensive machine breakdowns!"
print("✓ Cost optimization exercise passed.")
